# H&M Customer Segmentation — Phase 9
## Final Evaluation, Research Conclusions & Report Asset Export

Phase 9 **không tạo model mới**. Nó tổng hợp bằng chứng từ Phase 1–8 để trả lời Research Questions, ghi limitations/reproducibility và xuất bộ tài sản cuối cho báo cáo BTL.

### Research Questions
- **RQ1:** RFM baseline có đủ không?
- **RQ2:** Extended features có cải thiện clustering không?
- **RQ3:** Feature nào hữu ích và không quá redundant?
- **RQ4:** Số cluster nào phù hợp?
- **RQ5:** K-Means hay GMM phù hợp hơn?
- **RQ6:** Cluster có đủ khác biệt và đủ diễn giải để trở thành business segment không?
- **RQ7:** Segment hỗ trợ marketing decision support như thế nào?

Output chính:

```text
reports/final/
├── tables/
├── figures/
├── final_findings.md
├── methodology_summary.md
├── limitations.md
└── reproducibility_checklist.md
```

In [1]:
from pathlib import Path
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 240)
pd.set_option("display.width", 260)

ROOT = Path("..")
REPORTS = ROOT / "reports"
FEATURE_DIR = ROOT / "data" / "features"
RESULT_DIR = ROOT / "data" / "results"

FINAL_DIR = REPORTS / "final"
FINAL_TABLE = FINAL_DIR / "tables"
FINAL_FIG = FINAL_DIR / "figures"
for d in [FINAL_DIR, FINAL_TABLE, FINAL_FIG]: d.mkdir(parents=True, exist_ok=True)

ARTIFACTS = {
    "p2_cleaning": REPORTS/"tables"/"phase2_cleaning"/"14_cleaning_decision_log.csv",
    "p4_dictionary": REPORTS/"tables"/"phase4_features"/"10_feature_dictionary.csv",
    "p5_decision": REPORTS/"tables"/"phase5_feature_selection"/"17_feature_selection_decision_table.csv",
    "p5_forward": REPORTS/"tables"/"phase5_feature_selection"/"14_forward_feature_experiments.csv",
    "p5_single": REPORTS/"tables"/"phase5_feature_selection"/"13_single_addition_experiments.csv",
    "p6_compare": REPORTS/"tables"/"phase6_model_selection"/"12_kmeans_vs_gmm.csv",
    "p6_kmeans": REPORTS/"tables"/"phase6_model_selection"/"06_kmeans_by_k.csv",
    "p6_gmm": REPORTS/"tables"/"phase6_model_selection"/"08_gmm_by_k.csv",
    "p6_stability": REPORTS/"tables"/"phase6_model_selection"/"11_stability_summary.csv",
    "p7_segments": REPORTS/"tables"/"phase7_cluster_profiling"/"15_business_segment_summary.csv",
    "p7_effect": REPORTS/"tables"/"phase7_cluster_profiling"/"08_kruskal_wallis_effect_size.csv",
    "p7_interp": REPORTS/"tables"/"phase7_cluster_profiling"/"16_segment_interpretability_check.csv",
    "p8_strategy": REPORTS/"tables"/"phase8_marketing_decision_support"/"06_report_marketing_strategy_matrix.csv",
    "p8_objectives": REPORTS/"tables"/"phase8_marketing_decision_support"/"03_marketing_objective_summary.csv",
    "p8_guardrails": REPORTS/"tables"/"phase8_marketing_decision_support"/"05_decision_support_guardrails.csv",
    "p6_feature_set": FEATURE_DIR/"phase6_candidate_feature_set.csv",
    "segment_assignments": RESULT_DIR/"customer_segment_assignments.parquet"
}

audit=pd.DataFrame([{"artifact":k,"path":str(v),"status":"OK" if v.exists() else "MISSING"} for k,v in ARTIFACTS.items()])
display(audit)
audit.to_csv(FINAL_TABLE/"00_artifact_audit.csv", index=False)

def read_csv_safe(key):
    p=ARTIFACTS[key]
    return pd.read_csv(p) if p.exists() else None

p5_decision=read_csv_safe("p5_decision")
p5_forward=read_csv_safe("p5_forward")
p5_single=read_csv_safe("p5_single")
p6_compare=read_csv_safe("p6_compare")
p7_segments=read_csv_safe("p7_segments")
p7_effect=read_csv_safe("p7_effect")
p7_interp=read_csv_safe("p7_interp")
p8_strategy=read_csv_safe("p8_strategy")
p8_objectives=read_csv_safe("p8_objectives")

,artifact,path,status
0,p2_cleaning,..\reports\tables\phase2_cleaning\14_cleaning_...,OK
1,p4_dictionary,..\reports\tables\phase4_features\10_feature_d...,OK
2,p5_decision,..\reports\tables\phase5_feature_selection\17_...,OK
3,p5_forward,..\reports\tables\phase5_feature_selection\14_...,OK
4,p5_single,..\reports\tables\phase5_feature_selection\13_...,OK
5,p6_compare,..\reports\tables\phase6_model_selection\12_km...,OK
6,p6_kmeans,..\reports\tables\phase6_model_selection\06_km...,OK
7,p6_gmm,..\reports\tables\phase6_model_selection\08_gm...,OK
8,p6_stability,..\reports\tables\phase6_model_selection\11_st...,OK
9,p7_segments,..\reports\tables\phase7_cluster_profiling\15_...,OK


# 1. RFM Baseline vs Extended Feature Set

Dùng Phase 5 vì các feature-set experiments được chạy trên cùng sample, cùng `SCREEN_K` và cùng preprocessing rule. Đây là bằng chứng chính cho RQ1–RQ2.

In [2]:
baseline_vs_extended=None
if p5_forward is not None and len(p5_forward):
    f=p5_forward.copy()
    base=f.iloc[0]
    if "composite_rank" in f.columns:
        best=f.sort_values(["composite_rank","silhouette"], ascending=[True,False]).iloc[0]
    else:
        best=f.sort_values(["silhouette","davies_bouldin"], ascending=[False,True]).iloc[0]
    baseline_vs_extended=pd.DataFrame([
        {"feature_set":"RFM_BASELINE","features":base["features"],"n_features":base["n_features"],"silhouette":base["silhouette"],"davies_bouldin":base["davies_bouldin"],"calinski_harabasz":base["calinski_harabasz"]},
        {"feature_set":"BEST_EXTENDED","features":best["features"],"n_features":best["n_features"],"silhouette":best["silhouette"],"davies_bouldin":best["davies_bouldin"],"calinski_harabasz":best["calinski_harabasz"]}
    ])
elif p5_single is not None and len(p5_single):
    f=p5_single.copy(); base=f[f["experiment"]=="RFM_BASELINE"].iloc[0]
    best=f.sort_values(["silhouette","davies_bouldin"], ascending=[False,True]).iloc[0]
    baseline_vs_extended=pd.DataFrame([
        {"feature_set":"RFM_BASELINE","features":base["features"],"n_features":base["n_features"],"silhouette":base["silhouette"],"davies_bouldin":base["davies_bouldin"],"calinski_harabasz":base["calinski_harabasz"]},
        {"feature_set":"BEST_EXTENDED","features":best["features"],"n_features":best["n_features"],"silhouette":best["silhouette"],"davies_bouldin":best["davies_bouldin"],"calinski_harabasz":best["calinski_harabasz"]}
    ])

if baseline_vs_extended is not None:
    for m in ["silhouette","davies_bouldin","calinski_harabasz"]:
        baseline_vs_extended[f"delta_{m}_vs_rfm"] = baseline_vs_extended[m]-baseline_vs_extended.iloc[0][m]
    display(baseline_vs_extended)
    baseline_vs_extended.to_csv(FINAL_TABLE/"01_rfm_vs_extended.csv",index=False)

    fig,ax=plt.subplots(figsize=(8,5))
    ax.bar(baseline_vs_extended["feature_set"],baseline_vs_extended["silhouette"])
    ax.set_ylabel("Silhouette"); ax.set_title("RFM baseline vs Extended feature set")
    fig.tight_layout(); fig.savefig(FINAL_FIG/"01_rfm_vs_extended_silhouette.png",dpi=180,bbox_inches="tight"); plt.show()
else:
    print("Missing Phase 5 experiment evidence.")

,feature_set,features,n_features,silhouette,davies_bouldin,calinski_harabasz,delta_silhouette_vs_rfm,delta_davies_bouldin_vs_rfm,delta_calinski_harabasz_vs_rfm
0,RFM_BASELINE,recency_days | frequency_purchase_days | monet...,6,0.502425,0.743111,73480.682744,0.0,0.0,0.0
1,BEST_EXTENDED,recency_days | frequency_purchase_days | monet...,6,0.502425,0.743111,73480.682744,0.0,0.0,0.0


C:\Users\Dell\AppData\Local\Temp\ipykernel_27352\4199196537.py:30: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout(); fig.savefig(FINAL_FIG/"01_rfm_vs_extended_silhouette.png",dpi=180,bbox_inches="tight"); plt.show()


# 2. Final Feature-Selection Summary

Bảng này trả lời “vì sao giữ feature A nhưng bỏ feature B?” bằng cách giữ lại quality, ranking và clustering contribution.

In [3]:
final_feature_summary=None
if p5_decision is not None:
    cols=[c for c in ["feature","coverage_pct","quality_status","anova_rank","mi_rank","rf_rank","consensus_rank","single_delta_silhouette","single_delta_dbi","single_delta_ch","phase6_selected","decision"] if c in p5_decision.columns]
    final_feature_summary=p5_decision[cols].copy()
    display(final_feature_summary)
    final_feature_summary.to_csv(FINAL_TABLE/"02_final_feature_selection_summary.csv",index=False)

,feature,coverage_pct,quality_status,anova_rank,mi_rank,rf_rank,consensus_rank,single_delta_silhouette,single_delta_dbi,single_delta_ch,phase6_selected,decision
0,frequency_purchase_days,100.000000,CORE_KEEP,1.0,1.0,2.0,1.333333,NaN,NaN,NaN,True,KEEP_RFM_CORE
1,product_type_diversity,100.000000,OK,2.0,4.0,6.0,4.000000,0.004549,-0.022954,6971.833202,True,KEEP_FOR_PHASE6
2,product_diversity,100.000000,OK,5.0,6.0,3.0,4.666667,0.012830,-0.039817,7921.895541,True,KEEP_FOR_PHASE6
3,garment_group_diversity,100.000000,OK,3.0,3.0,10.0,5.333333,0.028033,-0.049691,9318.758581,True,KEEP_FOR_PHASE6
4,mean_interpurchase_days,67.063345,CAUTION,9.0,2.0,5.0,5.333333,-0.072509,0.161726,-20596.841761,False,NOT_SELECTED_AFTER_SCREENING
5,recency_days,100.000000,CORE_KEEP,7.0,9.0,1.0,5.666667,NaN,NaN,NaN,True,KEEP_RFM_CORE
6,monetary_total_spend,100.000000,CORE_KEEP,6.0,8.0,4.0,6.000000,NaN,NaN,NaN,True,KEEP_RFM_CORE
7,colour_diversity,100.000000,OK,4.0,7.0,11.0,7.333333,0.001555,-0.017188,5825.713953,False,NOT_SELECTED_AFTER_SCREENING
8,std_interpurchase_days,53.506165,CAUTION,10.0,5.0,9.0,8.000000,-0.055916,0.186402,-20507.246998,False,NOT_SELECTED_AFTER_SCREENING
9,avg_item_price,100.000000,OK,11.0,12.0,7.0,10.000000,-0.089821,0.184229,-22192.312945,False,NOT_SELECTED_AFTER_SCREENING


# 3. Final Model Selection

Tổng hợp K-Means vs GMM. AIC/BIC chỉ dùng nội bộ GMM; common comparison dùng Silhouette, DBI, CH và Stability ARI.

In [4]:
if p6_compare is not None:
    display(p6_compare)
    p6_compare.to_csv(FINAL_TABLE/"03_kmeans_vs_gmm_final.csv",index=False)
    fig,ax=plt.subplots(figsize=(8,5)); x=np.arange(len(p6_compare)); w=.35
    ax.bar(x-w/2,p6_compare["silhouette"],w,label="Silhouette")
    ax.bar(x+w/2,p6_compare["stability_ari_mean"],w,label="Stability ARI")
    ax.set_xticks(x); ax.set_xticklabels(p6_compare["algorithm"]); ax.set_title("Final algorithm comparison"); ax.legend()
    fig.tight_layout(); fig.savefig(FINAL_FIG/"02_final_algorithm_comparison.png",dpi=180,bbox_inches="tight"); plt.show()

,algorithm,k,silhouette,davies_bouldin,calinski_harabasz,stability_ari_mean,min_cluster_pct,max_cluster_pct,fit_seconds,rank_sil,rank_dbi,rank_ch,rank_stability,technical_composite_rank
0,KMeans,4,0.426303,0.790040,148614.495944,0.990677,11.135,38.482,0.772358,1.0,1.0,1.0,2.0,1.25
1,GMM,2,0.386070,0.948171,85472.756853,0.992890,44.888,55.112,3.563890,2.0,2.0,2.0,1.0,1.75


C:\Users\Dell\AppData\Local\Temp\ipykernel_27352\54292209.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout(); fig.savefig(FINAL_FIG/"02_final_algorithm_comparison.png",dpi=180,bbox_inches="tight"); plt.show()


# 4. Final Business Segments

Tổng hợp segment size, observed spending contribution và các feature khác biệt có effect size lớn.

In [5]:
if p7_segments is not None:
    display(p7_segments)
    p7_segments.to_csv(FINAL_TABLE/"04_final_business_segments.csv",index=False)

if p7_effect is not None:
    cols=[c for c in ["feature","epsilon_squared","effect_size","fdr_q_value","statistically_significant_fdr_05"] if c in p7_effect.columns]
    top_effects=p7_effect[cols].sort_values("epsilon_squared",ascending=False).head(15)
    display(top_effects)
    top_effects.to_csv(FINAL_TABLE/"05_top_between_cluster_effects.csv",index=False)

if p7_segments is not None and {"proposed_segment_name","pct_customers","pct_observed_spend"}.issubset(p7_segments.columns):
    d=p7_segments.copy(); x=np.arange(len(d)); w=.38
    fig,ax=plt.subplots(figsize=(11,6))
    ax.bar(x-w/2,d["pct_customers"],w,label="Customer share")
    ax.bar(x+w/2,d["pct_observed_spend"],w,label="Observed spend share")
    ax.set_xticks(x); ax.set_xticklabels(d["proposed_segment_name"],rotation=25,ha="right")
    ax.set_ylabel("Share (%)"); ax.set_title("Final segments — customer vs spending share"); ax.legend()
    fig.tight_layout(); fig.savefig(FINAL_FIG/"03_final_segment_value_distribution.png",dpi=180,bbox_inches="tight"); plt.show()

,cluster_id,base_segment_name,segment_descriptor,duplicate_name_flag,proposed_segment_name,age_state,avg_item_price_state,avg_spend_per_purchase_day_state,channel_diversity_state,colour_diversity_state,customer_tenure_days_state,department_diversity_state,dominant_channel_share_state,frequency_purchase_days_state,garment_group_diversity_state,item_frequency_state,items_per_purchase_day_state,monetary_total_spend_state,product_diversity_state,product_type_diversity_state,recency_days_state,section_diversity_state,customers,pct_customers,pct_observed_spend,spend_share_index,recency_days_robust_score,frequency_purchase_days_robust_score,monetary_total_spend_robust_score,recency_days,frequency_purchase_days,monetary_total_spend
0,0,Active Developing,No strong extended descriptor,False,Active Developing,Medium,Medium,Medium,Medium,Medium,Medium,Medium,Medium,Medium,Medium,Medium,Medium,Medium,Medium,Medium,Moderately Active,Medium,526357,38.637917,11.964120,0.309647,-0.071429,-0.142857,-0.143667,127.0,2.0,0.157508
1,1,High-Value Loyal,Broad Fashion Interest,False,High-Value Loyal,Medium,Medium,High,High,High,High,High,Low,High,High,High,Medium,High,High,High,Moderately Active,High,149398,10.966754,52.122944,4.752814,-0.360000,2.857143,3.532123,26.0,23.0,2.411203
2,2,Dormant Low-Value,No strong extended descriptor,False,Dormant Low-Value,Medium,Medium,Medium,Medium,Medium,Medium,Medium,Medium,Medium,Medium,Medium,Medium,Medium,Medium,Medium,Inactive,Medium,374073,27.459313,5.794771,0.211031,1.151429,-0.285714,-0.251396,555.0,1.0,0.091458
3,3,Frequent Core,Broad Fashion Interest,False,Frequent Core,Medium,Medium,Medium,High,High,High,High,Low,High,High,High,Medium,High,High,High,Moderately Active,High,312453,22.936017,30.118165,1.313138,-0.251429,0.857143,0.859402,64.0,9.0,0.772508


,feature,epsilon_squared,effect_size,fdr_q_value,statistically_significant_fdr_05
0,product_diversity,0.837072,large,0.0,True
1,department_diversity,0.832286,large,0.0,True
2,item_frequency,0.829996,large,0.0,True
3,product_type_diversity,0.819822,large,0.0,True
4,garment_group_diversity,0.817234,large,0.0,True
5,frequency_purchase_days,0.813939,large,0.0,True
6,monetary_total_spend,0.812090,large,0.0,True
7,section_diversity,0.807958,large,0.0,True
8,colour_diversity,0.796177,large,0.0,True
9,customer_tenure_days,0.700157,large,0.0,True


C:\Users\Dell\AppData\Local\Temp\ipykernel_27352\814470446.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout(); fig.savefig(FINAL_FIG/"03_final_segment_value_distribution.png",dpi=180,bbox_inches="tight"); plt.show()


# 5. Final Marketing Decision-Support Summary

Phase 8 là decision support, không phải causal optimal policy.

In [6]:
if p8_strategy is not None:
    display(p8_strategy)
    p8_strategy.to_csv(FINAL_TABLE/"06_final_marketing_strategy_matrix.csv",index=False)
if p8_objectives is not None:
    display(p8_objectives)
    p8_objectives.to_csv(FINAL_TABLE/"07_marketing_objective_distribution.csv",index=False)

,proposed_segment_name,segment_descriptor,business_diagnosis,marketing_objective,primary_action_family,secondary_action_family,message_direction,primary_kpi,secondary_kpi,risk_note,interpretable_for_phase8
0,Active Developing,No strong extended descriptor,Currently active customers who have not yet re...,Nurture and progression,Relevant product/category discovery,Cross-sell / repeat-purchase activation,Use known fashion preferences without aggressi...,Repeat purchase rate,Frequency / category breadth growth,This is a transitional segment and should be r...,True
1,Dormant Low-Value,No strong extended descriptor,Low historical value/frequency with low recent...,Low-cost re-engagement or deprioritization,Low-cost broad re-engagement,Experimental value proposition,Use low-cost channels before expensive incenti...,Reactivation rate,Incremental revenue per contacted customer,Avoid high spend on a low-response segment wit...,True
2,Frequent Core,Broad Fashion Interest,Strong purchase frequency but not yet exceptio...,Value expansion,Cross-sell / basket expansion,Premium-category discovery,Build on existing shopping habit with adjacent...,Spend per purchase day,Category breadth / average item price,Do not push premium products when price behavi...,True
3,High-Value Loyal,Broad Fashion Interest,Active customers with strong historical freque...,Retention and value protection,Loyalty / exclusivity / early-access engagement,Cross-sell into adjacent fashion categories,"Recognition, exclusivity, relevance; avoid unn...",Repeat purchase rate / retention rate,Category breadth / spend per active customer,Over-discounting high-value customers can dest...,True


,marketing_objective,customers,observed_spend,pct_customers,pct_observed_spend
0,Low-cost re-engagement or deprioritization,374073,51263.209000,27.459313,5.794771
1,Nurture and progression,526357,105840.105542,38.637917,11.964120
2,Retention and value protection,149398,461103.526915,10.966754,52.122944
3,Value expansion,312453,266439.132593,22.936017,30.118165


# 6. Research Question → Evidence → Conclusion

Notebook chỉ kết luận những gì artifacts thực tế hỗ trợ.

In [7]:
rq=[]

# RQ1–RQ2
if baseline_vs_extended is not None:
    a,b=baseline_vs_extended.iloc[0],baseline_vs_extended.iloc[1]
    wins=int(b["silhouette"]>a["silhouette"])+int(b["davies_bouldin"]<a["davies_bouldin"])+int(b["calinski_harabasz"]>a["calinski_harabasz"])
    rq.append({"research_question":"RQ1 — Is RFM baseline sufficient?","evidence":f"RFM Sil={a['silhouette']:.4f}, DBI={a['davies_bouldin']:.4f}, CH={a['calinski_harabasz']:.2f}; Extended Sil={b['silhouette']:.4f}, DBI={b['davies_bouldin']:.4f}, CH={b['calinski_harabasz']:.2f}.","conclusion":f"RFM is a valid baseline; the Extended set improved {wins}/3 screening metrics."})
    rq.append({"research_question":"RQ2 — Do Extended features improve segmentation?","evidence":f"Best Extended set: {b['features']}; improvement count={wins}/3.","conclusion":"Extended features show incremental clustering value." if wins>=2 else "Extended features did not consistently improve all clustering metrics."})
else:
    rq += [{"research_question":"RQ1 — Is RFM baseline sufficient?","evidence":"Missing Phase 5 comparison.","conclusion":"Insufficient evidence."},{"research_question":"RQ2 — Do Extended features improve segmentation?","evidence":"Missing Phase 5 comparison.","conclusion":"Insufficient evidence."}]

# RQ3
if final_feature_summary is not None:
    if "phase6_selected" in final_feature_summary.columns:
        sel=final_feature_summary[final_feature_summary["phase6_selected"]==True]["feature"].astype(str).tolist()
    else: sel=[]
    rq.append({"research_question":"RQ3 — Which features are useful and non-redundant?","evidence":"Quality + redundancy + ANOVA + MI + Random Forest + consensus + forward/ablation were combined.","conclusion":"Selected candidate features: "+(", ".join(sel) if sel else "see Phase 5 decision table")})

# RQ4–RQ5
if p6_compare is not None and len(p6_compare):
    w=p6_compare.sort_values(["technical_composite_rank","stability_ari_mean","silhouette"],ascending=[True,False,False]).iloc[0]
    rq.append({"research_question":"RQ4 — What number of clusters/components is appropriate?","evidence":"K-Means and GMM were evaluated for 2–8 clusters/components using internal metrics, balance and stability; GMM additionally used AIC/BIC.","conclusion":f"Technical winner uses {int(w['k'])} clusters/components."})
    rq.append({"research_question":"RQ5 — K-Means or GMM?","evidence":f"Winner={w['algorithm']}; Sil={w['silhouette']:.4f}; DBI={w['davies_bouldin']:.4f}; CH={w['calinski_harabasz']:.2f}; ARI={w['stability_ari_mean']:.4f}.","conclusion":f"{w['algorithm']} selected with k={int(w['k'])}."})

# RQ6
if p7_segments is not None:
    medlarge=0
    if p7_effect is not None and "effect_size" in p7_effect.columns:
        medlarge=int(p7_effect["effect_size"].isin(["medium","large"]).sum())
    rq.append({"research_question":"RQ6 — Are clusters meaningful enough for business interpretation?","evidence":f"{len(p7_segments)} segment profiles; {medlarge} features have medium/large effect size.","conclusion":"Segments are interpreted using measured R/F/M states, Extended descriptors, spending contribution and statistical profile differences."})

# RQ7
if p8_strategy is not None:
    rq.append({"research_question":"RQ7 — How can segments support marketing decisions?","evidence":f"{len(p8_strategy)} strategy rows map segment profiles to diagnosis, objective, action family, KPI and risk.","conclusion":"Segments support differentiated marketing decision support, but actions remain hypotheses requiring controlled evaluation."})

rq_matrix=pd.DataFrame(rq)
display(rq_matrix)
rq_matrix.to_csv(FINAL_TABLE/"08_research_question_evidence_matrix.csv",index=False)

,research_question,evidence,conclusion
0,RQ1 — Is RFM baseline sufficient?,"RFM Sil=0.5024, DBI=0.7431, CH=73480.68; Exten...",RFM is a valid baseline; the Extended set impr...
1,RQ2 — Do Extended features improve segmentation?,Best Extended set: recency_days | frequency_pu...,Extended features did not consistently improve...
2,RQ3 — Which features are useful and non-redund...,Quality + redundancy + ANOVA + MI + Random For...,Selected candidate features: frequency_purchas...
3,RQ4 — What number of clusters/components is ap...,K-Means and GMM were evaluated for 2–8 cluster...,Technical winner uses 4 clusters/components.
4,RQ5 — K-Means or GMM?,Winner=KMeans; Sil=0.4263; DBI=0.7900; CH=1486...,KMeans selected with k=4.
5,RQ6 — Are clusters meaningful enough for busin...,4 segment profiles; 15 features have medium/la...,Segments are interpreted using measured R/F/M ...
6,RQ7 — How can segments support marketing decis...,4 strategy rows map segment profiles to diagno...,Segments support differentiated marketing deci...


# 7. Limitations & Reproducibility

Hai phần này phải xuất hiện trong báo cáo để tránh diễn giải quá mức.

In [8]:
limitations=pd.DataFrame([
    ["Dataset period","Historical H&M observation period, not current 2026 behavior.","Interpret as methodology validation on this dataset."],
    ["No order_id","Exact order count is unavailable.","Define Frequency as distinct purchase days."],
    ["No quantity field","Repeated rows may represent multiple units.","Do not mechanically drop exact transaction duplicates."],
    ["No browsing/cart events","CVR and cart-abandonment cannot be measured.","Exclude rather than fabricate proxies."],
    ["No returns/cancellation status","ReturnRate/CancelRate unavailable.","Do not claim these dimensions are included."],
    ["No campaign-treatment data","Marketing impact cannot be estimated causally.","Use A/B tests, uplift or causal inference for treatment effectiveness."],
    ["Unsupervised clustering","No ground-truth segment labels.","Use multiple metrics, stability, profiling and interpretability."],
    ["Cluster naming","Segment names are interpretations, not true labels.","Keep naming rules auditable and preserve cluster IDs."],
    ["FuturePurchase ranking target","This is a proxy business outcome for ranking, not the clustering target.","Combine supervised ranking with unsupervised clustering utility."],
    ["Model drift","Segment structure may change over time.","Re-audit and retrain after major behavior shifts."]
],columns=["limitation","implication","mitigation"])
limitations.to_csv(FINAL_TABLE/"09_limitations.csv",index=False)
display(limitations)

repro=pd.DataFrame([
    ["Dataset fixed","Same H&M snapshot for all experiments."],
    ["Raw immutable","Keep raw CSV files unchanged."],
    ["Cleaning logged","Use Phase 2 cleaning decision log."],
    ["Reference date fixed","Derive RFM reference date deterministically."],
    ["Random seed fixed","Use RANDOM_STATE=42 unless documented otherwise."],
    ["Feature definitions documented","Use Phase 4 feature dictionary."],
    ["Future split documented","Phase 5 uses a fixed 30-day FuturePurchase window."],
    ["Same screening samples","Feature-set/model comparisons reuse fixed samples."],
    ["Preprocessor/model saved","Persist final transformations and model."],
    ["Naming auditable","Keep Phase 7 naming evidence."],
    ["Marketing guardrails documented","Keep Phase 8 decision-support limits."]
],columns=["item","rule"])
repro.to_csv(FINAL_TABLE/"10_reproducibility_checklist.csv",index=False)
display(repro)

,limitation,implication,mitigation
0,Dataset period,"Historical H&M observation period, not current...",Interpret as methodology validation on this da...
1,No order_id,Exact order count is unavailable.,Define Frequency as distinct purchase days.
2,No quantity field,Repeated rows may represent multiple units.,Do not mechanically drop exact transaction dup...
3,No browsing/cart events,CVR and cart-abandonment cannot be measured.,Exclude rather than fabricate proxies.
4,No returns/cancellation status,ReturnRate/CancelRate unavailable.,Do not claim these dimensions are included.
5,No campaign-treatment data,Marketing impact cannot be estimated causally.,"Use A/B tests, uplift or causal inference for ..."
6,Unsupervised clustering,No ground-truth segment labels.,"Use multiple metrics, stability, profiling and..."
7,Cluster naming,"Segment names are interpretations, not true la...",Keep naming rules auditable and preserve clust...
8,FuturePurchase ranking target,"This is a proxy business outcome for ranking, ...",Combine supervised ranking with unsupervised c...
9,Model drift,Segment structure may change over time.,Re-audit and retrain after major behavior shifts.


,item,rule
0,Dataset fixed,Same H&M snapshot for all experiments.
1,Raw immutable,Keep raw CSV files unchanged.
2,Cleaning logged,Use Phase 2 cleaning decision log.
3,Reference date fixed,Derive RFM reference date deterministically.
4,Random seed fixed,Use RANDOM_STATE=42 unless documented otherwise.
5,Feature definitions documented,Use Phase 4 feature dictionary.
6,Future split documented,Phase 5 uses a fixed 30-day FuturePurchase win...
7,Same screening samples,Feature-set/model comparisons reuse fixed samp...
8,Preprocessor/model saved,Persist final transformations and model.
9,Naming auditable,Keep Phase 7 naming evidence.


# 8. Methodology Diagram & Final Figure Manifest

In [9]:
stages=["1 Audit","2 Cleaning","3 EDA","4 RFM + Extended","5 Feature Selection","6 K-Means vs GMM","7 Profiling + Naming","8 Marketing Support","9 Final Evaluation"]
fig,ax=plt.subplots(figsize=(16,3)); ax.axis("off"); xs=np.linspace(.05,.95,len(stages))
for i,(x,label) in enumerate(zip(xs,stages)):
    ax.text(x,.5,label,ha="center",va="center",transform=ax.transAxes,bbox=dict(boxstyle="round,pad=.45",fc="white",ec="black"))
    if i<len(stages)-1:
        ax.annotate("",xy=(xs[i+1]-.045,.5),xytext=(x+.045,.5),xycoords=ax.transAxes,textcoords=ax.transAxes,arrowprops=dict(arrowstyle="->"))
ax.set_title("Research Pipeline — H&M Customer Segmentation & Marketing Decision Support",pad=18)
fig.tight_layout(); fig.savefig(FINAL_FIG/"04_research_methodology_pipeline.png",dpi=180,bbox_inches="tight"); plt.show()

selected=[
    REPORTS/"figures"/"phase3_eda"/"01_monthly_transaction_rows.png",
    REPORTS/"figures"/"phase3_eda"/"05_purchase_days_distribution.png",
    REPORTS/"figures"/"phase4_features"/"09_candidate_feature_correlation.png",
    REPORTS/"figures"/"phase5_feature_selection"/"07_consensus_feature_ranking.png",
    REPORTS/"figures"/"phase5_feature_selection"/"13_forward_composite_rank.png",
    REPORTS/"figures"/"phase6_model_selection"/"02_kmeans_elbow.png",
    REPORTS/"figures"/"phase6_model_selection"/"08_gmm_aic_bic.png",
    REPORTS/"figures"/"phase6_model_selection"/"11_model_stability_ari.png",
    REPORTS/"figures"/"phase6_model_selection"/"14_final_model_pca.png",
    REPORTS/"figures"/"phase7_cluster_profiling"/"02_cluster_profile_heatmap.png",
    REPORTS/"figures"/"phase7_cluster_profiling"/"04_feature_effect_sizes.png",
    REPORTS/"figures"/"phase8_marketing_decision_support"/"01_marketing_objective_customer_share.png"
]
manifest=[]
for src in selected:
    if src.exists():
        dst=FINAL_FIG/src.name; shutil.copy2(src,dst); status="COPIED"
    else:
        dst=""; status="MISSING"
    manifest.append({"source":str(src),"final_path":str(dst),"status":status})
manifest=pd.DataFrame(manifest); display(manifest); manifest.to_csv(FINAL_TABLE/"11_final_figure_manifest.csv",index=False)

C:\Users\Dell\AppData\Local\Temp\ipykernel_27352\1170419504.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout(); fig.savefig(FINAL_FIG/"04_research_methodology_pipeline.png",dpi=180,bbox_inches="tight"); plt.show()


,source,final_path,status
0,..\reports\figures\phase3_eda\01_monthly_trans...,..\reports\final\figures\01_monthly_transactio...,COPIED
1,..\reports\figures\phase3_eda\05_purchase_days...,..\reports\final\figures\05_purchase_days_dist...,COPIED
2,..\reports\figures\phase4_features\09_candidat...,..\reports\final\figures\09_candidate_feature_...,COPIED
3,..\reports\figures\phase5_feature_selection\07...,..\reports\final\figures\07_consensus_feature_...,COPIED
4,..\reports\figures\phase5_feature_selection\13...,..\reports\final\figures\13_forward_composite_...,COPIED
5,..\reports\figures\phase6_model_selection\02_k...,..\reports\final\figures\02_kmeans_elbow.png,COPIED
6,..\reports\figures\phase6_model_selection\08_g...,..\reports\final\figures\08_gmm_aic_bic.png,COPIED
7,..\reports\figures\phase6_model_selection\11_m...,..\reports\final\figures\11_model_stability_ar...,COPIED
8,..\reports\figures\phase6_model_selection\14_f...,..\reports\final\figures\14_final_model_pca.png,COPIED
9,..\reports\figures\phase7_cluster_profiling\02...,..\reports\final\figures\02_cluster_profile_he...,COPIED


# 9. Export Final Markdown Findings

Các file Markdown này là bản tổng hợp để viết Chương 4–7 của BTL.

In [10]:
lines=["# Final Findings — H&M Customer Segmentation","","## Research Questions",""]
for _,r in rq_matrix.iterrows():
    lines += [f"### {r['research_question']}","",f"**Evidence:** {r['evidence']}","",f"**Conclusion:** {r['conclusion']}",""]

lines += ["## Final Feature Set",""]
if ARTIFACTS["p6_feature_set"].exists():
    fs=pd.read_csv(ARTIFACTS["p6_feature_set"]); lines.append(", ".join(fs["feature"].astype(str).tolist()))
else: lines.append("Feature-set artifact missing.")

lines += ["","## Final Model",""]
if p6_compare is not None and len(p6_compare):
    w=p6_compare.sort_values(["technical_composite_rank","stability_ari_mean","silhouette"],ascending=[True,False,False]).iloc[0]
    lines += [f"**{w['algorithm']}**, k/components={int(w['k'])}.","",f"Silhouette={w['silhouette']:.4f}; DBI={w['davies_bouldin']:.4f}; CH={w['calinski_harabasz']:.2f}; Stability ARI={w['stability_ari_mean']:.4f}."]

lines += ["","## Final Segments",""]
if p7_segments is not None:
    for _,r in p7_segments.iterrows():
        lines.append(f"- **{r.get('proposed_segment_name','Unknown')}** — customers={r.get('pct_customers','N/A')}%; observed spend={r.get('pct_observed_spend','N/A')}%; descriptor={r.get('segment_descriptor','')}")

lines += ["","## Marketing Decision Support",""]
if p8_strategy is not None:
    for _,r in p8_strategy.iterrows():
        lines.append(f"- **{r.get('proposed_segment_name','Unknown')}** → {r.get('marketing_objective','N/A')} → {r.get('primary_action_family','N/A')}")

lines += ["","## Limitations",""]
for _,r in limitations.iterrows():
    lines.append(f"- **{r['limitation']}**: {r['implication']} Mitigation: {r['mitigation']}")

lines += ["","## Final Interpretation","","The study should be interpreted as a reproducible customer-segmentation and marketing-decision-support methodology evaluated on the H&M public dataset. Segment names are business interpretations of measured historical behavior, not ground-truth labels. Marketing actions remain hypotheses to test rather than causally optimal policies."]

(FINAL_DIR/"final_findings.md").write_text("\n".join(lines), encoding="utf-8")

methodology="""# Methodology Summary

1. Dataset Audit
2. Cleaning Decisions
3. Exploratory Data Analysis
4. RFM + Extended Feature Engineering
5. Feature Selection
6. K-Means vs GMM Model Selection
7. Cluster Profiling + Statistical Validation + Naming
8. Marketing Decision Support
9. Final Evaluation
"""
(FINAL_DIR/"methodology_summary.md").write_text(methodology,encoding="utf-8")

lim=["# Limitations",""]
for _,r in limitations.iterrows(): lim += [f"## {r['limitation']}","",f"**Implication:** {r['implication']}","",f"**Mitigation:** {r['mitigation']}",""]
(FINAL_DIR/"limitations.md").write_text("\n".join(lim), encoding="utf-8")

rep=["# Reproducibility Checklist",""]+[f"- **{r['item']}** — {r['rule']}" for _,r in repro.iterrows()]
(FINAL_DIR/"reproducibility_checklist.md").write_text("\n".join(rep), encoding="utf-8")

print("Final report markdown assets exported to", FINAL_DIR)

Final report markdown assets exported to ..\reports\final


# 10. Cấu trúc báo cáo BTL đề xuất

```text
CHƯƠNG 1 — GIỚI THIỆU
1.1 Bối cảnh
1.2 Vấn đề nghiên cứu
1.3 Mục tiêu
1.4 Research Questions
1.5 Phạm vi

CHƯƠNG 2 — CƠ SỞ LÝ THUYẾT & RELATED WORK
2.1 Customer Segmentation
2.2 RFM / Extended RFM
2.3 Feature Selection
2.4 K-Means / GMM
2.5 Cluster Validation
2.6 Nghiên cứu liên quan

CHƯƠNG 3 — DỮ LIỆU VÀ PHƯƠNG PHÁP
3.1 H&M Dataset
3.2 Data Audit & Cleaning
3.3 EDA
3.4 RFM Definition
3.5 Extended Feature Engineering
3.6 Feature Selection
3.7 Transformation & Scaling
3.8 K-Means / GMM
3.9 Evaluation Metrics

CHƯƠNG 4 — THỰC NGHIỆM VÀ KẾT QUẢ
4.1 Data Quality / EDA
4.2 Feature Selection
4.3 RFM vs Extended Features
4.4 K-Means Selection
4.5 GMM Selection
4.6 Stability
4.7 Final Model
4.8 Cluster Profiles
4.9 Statistical Validation

CHƯƠNG 5 — SEGMENTS & MARKETING DECISION SUPPORT
5.1 Segment Naming
5.2 Segment Characteristics
5.3 Customer/Spending Contribution
5.4 Marketing Objectives
5.5 Individual Customer Demo
5.6 KPI & Guardrails

CHƯƠNG 6 — THẢO LUẬN
6.1 Trả lời RQ1–RQ7
6.2 So sánh Related Work
6.3 Ý nghĩa thực tiễn
6.4 Limitations / Threats to Validity

CHƯƠNG 7 — KẾT LUẬN & HƯỚNG PHÁT TRIỂN
```

In [11]:
required=[
    FINAL_TABLE/"00_artifact_audit.csv",
    FINAL_TABLE/"08_research_question_evidence_matrix.csv",
    FINAL_TABLE/"09_limitations.csv",
    FINAL_TABLE/"10_reproducibility_checklist.csv",
    FINAL_DIR/"final_findings.md",
    FINAL_DIR/"methodology_summary.md",
    FINAL_DIR/"limitations.md",
    FINAL_DIR/"reproducibility_checklist.md",
    FINAL_FIG/"04_research_methodology_pipeline.png"
]
for p in required: assert p.exists(), f"Missing: {p}"
print("Phase 9 validation: PASSED")
print("Final assets directory:", FINAL_DIR)

Phase 9 validation: PASSED
Final assets directory: ..\reports\final


# Sau Phase 9

Pipeline nghiên cứu chính đã hoàn tất. Không nên thêm model/feature chỉ để bài trông phức tạp hơn.

Các việc tiếp theo hợp lý là:
1. Viết báo cáo BTL chính thức từ `reports/final/`.
2. Chọn khoảng 10–15 figure quan trọng nhất.
3. Viết Related Work và đối chiếu với các paper đã nghiên cứu.
4. Chuẩn bị slide thuyết trình.
5. Chuẩn bị câu hỏi phản biện về Frequency, Extended RFM, Feature Selection, K-Means/GMM, metrics, naming và Marketing Decision Support.